In [ ]:
import pandas as pd
import numpy as n
import matplotlib.pyplot as plt
import os
from sklearn.cluster import KMeans


In [ ]:
df = pd.read_csv('/Users/<user>/PhD/github/qoe-research/pcap_durations.csv')
df['duration'].sum()
# len(df)

In [ ]:
# Rating Quality Distortion
# 5 Excellent Imperceptible
# 4 Good Just perceptible, but not annoying
# 3 Fair Perceptible and slightly annoying
# 2 Poor Annoying, but not objectionable
# 1 Bad Very annoying and objectionable
def rank_qoe(ping):
    if ping <= 80:
        return 'Excellent'
    elif 80 < ping <= 120:
        return 'Good'
    elif 120 < ping <= 160:
        return 'Fair'
    elif 160 < ping < 200:
        return 'Poor'
    elif 200 >= ping:
        return 'Bad'
    return 'Error'

def rank_impact(impact):
    if impact <= 80:
        return 'Bad'
    elif 80 < impact <= 120:
        return 'Poor'
    elif 120 < impact <= 160:
        return 'Fair'
    elif 160 < impact < 200:
        return 'Good'
    elif 200 >= impact:
        return 'Excellent'
    return 'Error'

rank_str_to_int_mapping = {
    'Excellent': 5,
    'Good': 4,
    'Fair': 3,
    'Poor': 2,
    'Bad': 1
}

In [ ]:
df = pd.read_csv('/Users/<user>/phd/github/qoe-research/csv/window.csv')
# df = pd.read_csv('beni-csv/window.csv')

In [ ]:
exp_df = pd.read_csv('/Users/<user>/PhD/github/qoe-research/exp/10-Dec-2023/table.csv')

In [ ]:
exp_df['label'] = exp_df['avg_ping_rate'].apply(rank_qoe)

In [ ]:
exp_df.groupby(by=['label']).agg({
    'accuracy': ['median', 'mean'],
    'score': ['median', 'mean'],
    'self_rate_qoe_score': ['median', 'mean'],
    'kill': ['median', 'mean'],
    'death': ['median', 'mean', 'count'],
}).sort_values(by='label').dropna()

In [ ]:
impact_df = df.groupby(by=['exp_id', 'ping', 'impact']).agg({
    'src2dst_pps': 'median',
    'dst2src_pps': 'median',
    'src2dst_avg_pkt_size': 'median',
    'dst2src_avg_pkt_size': 'median',
}).sort_values(by='ping').reset_index()
impact_df

In [ ]:
analysis_df = df.groupby(by=['exp_id', 'ping', 'self_rate_qoe_score']).agg({
    'src2dst_pps': 'median',
    'dst2src_pps': 'median',
    'src2dst_avg_pkt_size': 'median',
    'dst2src_avg_pkt_size': 'median',
}).sort_values(by='ping').reset_index()

In [ ]:
exp7_df = df[df['exp_id'] == 10]
exp7_df

In [ ]:
# plot a trend line chart of src2dst_pps in red and dst2src_pps in blue vs. pktIdx starting for pktIdx 25 to 300
# make src2dst_pps dotted line
# make dst2src_pps solid line
# exp7_df.plot(x='pktIdx', y=['src2dst_pps_median', 'dst2src_pps_median'], color=['red', 'blue'], xlim=(25, 301), style=['o', 's'])
exp7_df.plot(x='pktIdx', y=['src2dst_pps', 'dst2src_pps'], color=['red', 'blue'], xlim=(25, 301), style=['o', 's'])
# exp7_df.plot(x='pktIdx', y=['src2dst_pps', 'dst2src_pps'], color=['red', 'blue'], xlim=(25, 300))

# y max value should be 200
plt.ylim(0, 200)
plt.legend().remove()
plt.xticks(range(25, 301, 25))
plt.xlabel('Gameplay Time')
plt.ylabel('PPS')
plt.title('PPS vs. Gameplay Time')
# plot src2dst_pps and dst2src_pps with plt instead of pandas plot vs pktIdx
# exp7_df.plot(x='pktIdx', y=['src2dst_pps', 'dst2src_pps'], color=['red', 'blue'])
#make the plot wider
plt.gcf().set_size_inches(12, 6)

In [ ]:
plt.xlabel('Subjective QoE Rating')
plt.ylabel('Ping Rate')
plt.title('Ping Rate vs. Subjective QoE Rating')

for i, row in analysis_df.iterrows():
    x, y1, label = row['self_rate_qoe_score'], row['ping'], row['exp_id']
    # x axis should show only int values
    plt.xticks(range(1,6))
    plt.scatter(x, y1, c='red', marker='o')
    # plt.scatter(x, y2, c='blue', marker='o')

In [ ]:
plt.xlabel('ACR')
plt.ylabel('PPS')
plt.title('PPS vs. ACR')

for i, row in analysis_df.iterrows():
    x, y1, y2, label = row['self_rate_qoe_score'], row['src2dst_pps'], row['dst2src_pps'], row['exp_id']
    # x axis should show only int values
    plt.xticks(range(1,6))
    plt.scatter(x, y1, c='red', marker='s')
    plt.scatter(x, y2, c='blue', marker='o')
    # plt.annotate(label, (x, y1), textcoords="offset points", xytext=(0, 0), ha='center', c='b')
    # plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='r')

In [ ]:
# plt.scatter(x='ping', y='src2dst_pps', label='exp_id', data=analysis_df, c='red', marker='s')
# plt.scatter(x='ping', y='dst2src_pps', label='exp_id', data=analysis_df, c='blue', marker='o')
plt.xlabel('Ping Rate')
plt.ylabel('PPS')
plt.title('PPS vs. Ping Rate')

for i, row in analysis_df.iterrows():
    x, y1, y2, label = row['ping'], row['src2dst_pps'], row['dst2src_pps'], row['exp_id']
    # plt.scatter(x, y1, c='blue')
    # plt.scatter(x, y2, c='red')
    plt.scatter(x, y1, c='red', marker='s')
    plt.scatter(x, y2, c='blue', marker='o')
    # plt.annotate(label, (x, y1), textcoords="offset points", xytext=(0, 0), ha='center', c='b')
    # plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='r')

plt.grid(True)

In [ ]:
plt.xlabel('Ping')
plt.ylabel('Packet Size')
plt.title('Ping vs. Average Packet Size')

for i, row in analysis_df.iterrows():
    x, y1, y2, label = row['ping'], row['src2dst_avg_pkt_size'], row['dst2src_avg_pkt_size'], row['exp_id']
    plt.scatter(x, y1, c='blue', label=label)
    plt.scatter(x, y2, c='red', label=label)
    # plt.annotate(label, (x, y1), textcoords="offset points", xytext=(0, 0), ha='center', c='b')
    # plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='r')

plt.grid(True)

In [ ]:
plt.xlabel('MOS')
plt.ylabel('Packet Size')
plt.title('MOS vs. Average Packet Size')

for i, row in analysis_df.iterrows():
    x, y1, y2, label = row['self_rate_qoe_score'], row['src2dst_avg_pkt_size'], row['dst2src_avg_pkt_size'], row['exp_id']
    plt.scatter(x, y1, c='blue', label=label)
    plt.scatter(x, y2, c='red', label=label)
    plt.annotate(label, (x, y1), textcoords="offset points", xytext=(0, 0), ha='center', c='b')
    plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='r')

In [ ]:
plt.xlabel('src2dst PPS')
plt.ylabel('dst2src PPS')
plt.title('Median PPS')

for i, row in analysis_df.iterrows():
    x, y, label1 =row['src2dst_pps'], row['dst2src_pps'], row['ping']
    plt.scatter(x, y, c='blue', label=label1, alpha=0.3)
    plt.annotate(label1, (x, y), textcoords="offset points", xytext=(0, 0), ha='center', c='b')
    # plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='r')

plt.grid(True)

In [ ]:
plt.xlabel('src2dst_avg_pkt_size')
plt.ylabel('dst2src_avg_pkt_size')
plt.title('Median PPS')

for i, row in analysis_df.iterrows():
    x, y, label1 =row['src2dst_avg_pkt_size'], row['dst2src_avg_pkt_size'], row['ping']
    plt.scatter(x, y, c='blue', label=label1, alpha=0.3)
    plt.annotate(label1, (x, y), textcoords="offset points", xytext=(0, 0), ha='center', c='b')
    # plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='r')

plt.grid(True)

In [ ]:
# plot scatter chart of self_rate_qoe_score vs ping rate on the dataset dataframe
plt.xlabel('self_rate_qoe_score')
plt.ylabel('ping')
plt.title('self_rate_qoe_score vs ping')

plt.scatter(df['self_rate_qoe_score'], df['ping'])


In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle

# Load the dataset
features = ['src2dst_avg_pkt_size', 'src2dst_avg_pkt_size_mean', 'src2dst_avg_pkt_size_median', 'src2dst_avg_pkt_size_std', 'src2dst_avg_pkt_size_min', 'src2dst_avg_pkt_size_max', 'src2dst_pps', 'src2dst_pps_mean', 'src2dst_pps_median', 'src2dst_pps_std', 'src2dst_pps_min', 'src2dst_pps_max', 'dst2src_avg_pkt_size', 'dst2src_avg_pkt_size_mean', 'dst2src_avg_pkt_size_median', 'dst2src_avg_pkt_size_std', 'dst2src_avg_pkt_size_min', 'dst2src_avg_pkt_size_max', 'dst2src_pps', 'dst2src_pps_mean', 'dst2src_pps_median', 'dst2src_pps_std', 'dst2src_pps_min', 'dst2src_pps_max', 'pkt_dir_ratio', 'pkt_dir_ratio_mean', 'pkt_dir_ratio_median', 'pkt_dir_ratio_std', 'pkt_dir_ratio_min', 'pkt_dir_ratio_max']
dataset = pd.read_csv('/Users/<user>/PhD/github/qoe-research/csv/window.csv') 
dataset.dropna(subset=features, inplace=True)
dataset['y_qoe_by_ping'] = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])
X_train = dataset[dataset['exp_id'] < 59][features]
y_train_by_ping = dataset[dataset['exp_id'] < 59]['y_qoe_by_ping']
X_test = dataset[dataset['exp_id'] >= 59][features]
y_test_by_ping = dataset[dataset['exp_id'] >= 59]['y_qoe_by_ping']

# X = dataset[features]
# y_qoe_by_ping = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])
# X_train, X_test, y_train_by_ping, y_test_by_ping = train_test_split(X, y_qoe_by_ping, test_size=0.4, random_state=42)
y_train_by_survey = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_test_by_survey = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_train_by_impact = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])
y_test_by_impact = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle

# Load the dataset
features = ['src2dst_avg_pkt_size', 'src2dst_avg_pkt_size_mean', 'src2dst_avg_pkt_size_median', 'src2dst_avg_pkt_size_std', 'src2dst_avg_pkt_size_min', 'src2dst_avg_pkt_size_max', 'src2dst_pps', 'src2dst_pps_mean', 'src2dst_pps_median', 'src2dst_pps_std', 'src2dst_pps_min', 'src2dst_pps_max', 'dst2src_avg_pkt_size', 'dst2src_avg_pkt_size_mean', 'dst2src_avg_pkt_size_median', 'dst2src_avg_pkt_size_std', 'dst2src_avg_pkt_size_min', 'dst2src_avg_pkt_size_max', 'dst2src_pps', 'dst2src_pps_mean', 'dst2src_pps_median', 'dst2src_pps_std', 'dst2src_pps_min', 'dst2src_pps_max', 'pkt_dir_ratio', 'pkt_dir_ratio_mean', 'pkt_dir_ratio_median', 'pkt_dir_ratio_std', 'pkt_dir_ratio_min', 'pkt_dir_ratio_max']
dataset = pd.read_csv('/Users/<user>/PhD/github/qoe-research/csv_5/window.csv') 
dataset.dropna(subset=features, inplace=True)
X = dataset[features]
y_qoe_by_ping = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])
X_train, X_test, y_train_by_ping, y_test_by_ping = train_test_split(X, y_qoe_by_ping, test_size=0.4, random_state=42)
y_train_by_survey = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_test_by_survey = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_train_by_impact = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])
y_test_by_impact = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])

In [ ]:
len(X_train), len(y_test_by_ping), len(X_train) + len(X_test)

In [ ]:
# have both y_qoe_by_ping.value_counts(), y_test_by_ping.value_counts() in one dataframe
ping_df = pd.DataFrame({'y_qoe_by_ping': y_qoe_by_ping.value_counts(), 'y_test_by_ping': y_test_by_ping.value_counts()})
qoe_df = pd.DataFrame({'y_qoe_by_survey': y_train_by_survey.value_counts(), 'y_test_by_survey': y_test_by_survey.value_counts()})

In [ ]:
ping_df

In [ ]:
qoe_df

In [ ]:
ping_clf = RandomForestClassifier()
survey_clf = RandomForestClassifier()
impact_clf = RandomForestClassifier()
# Train the models
ping_clf.fit(X_train, y_train_by_ping)
survey_clf.fit(X_train, y_train_by_survey)
impact_clf.fit(X_train, y_train_by_survey)

y_pred_by_ping = ping_clf.predict(X_test)
y_pred_by_survey = survey_clf.predict(X_test)
y_pred_by_impact = impact_clf.predict(X_test)


In [ ]:
X_train.iloc[1079]

In [ ]:
# print(len(X), len(X_train), len(X_test))
# print([v for v in y_pred])
# print(list(y_test))
y_test_list = list(y_pred_by_ping)
for i, _ in enumerate(y_test_list):
    print(y_test_list[i], y_pred_by_ping[i], y_pred_by_survey[i])
    # print(y_test_list[i], y_pred_qoe_by_ping[i], y_pred_qoe_by_survey[i])

# with open('model4.pkl', 'wb') as file:
#     pickle.dump(clf, file) 

In [ ]:
plot_confusion_matrix(y_test_by_ping, y_pred_by_ping)

In [ ]:
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(y_test_by_survey, y_pred_by_survey))
plot_confusion_matrix(y_test_by_survey, y_pred_by_survey)

In [ ]:
print(mean_absolute_error(y_test_by_ping, y_pred_by_ping))
plot_confusion_matrix(y_test_by_ping, y_pred_by_ping)

In [ ]:
plot_confusion_matrix(y_test_by_survey, y_pred_by_survey)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np
import seaborn as sns

def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    true_labels = ['Bad', 'Poor', 'Fair', 'Good', 'Excellent']
    # true_labels = ['Excellent', 'Good', 'Fair', 'Bad']
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=true_labels,
                yticklabels=true_labels)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    for i, data in enumerate(cm):
        plt.bar(range(len(data)), data)

        # Set labels and title
        plt.xlabel('Category')
        plt.xticks(range(len(true_labels)), true_labels)
        plt.ylabel('Frequency')
        plt.title(f'True Label {true_labels[i]} Predications')

        # Show the plot
        plt.show()

In [ ]:

dir_name = '/Users/<user>/PhD/github/qoe-research/eyal-csv'
# filename = '20230611-HUMAN5v5_FPS60_#1_D_0_0_50_U_0_0_50.csv'
for filename in os.listdir(dir_name):
    df = pd.read_csv(os.path.join(dir_name, filename))
    df = df[df['pkt_dir_ratio'] > 0.1]

    # Create some mock data
    t = df['pktIdx']
    data1 = df['src2dst_pps'].rolling(window=10).max()
    data2 = df['dst2src_pps'].rolling(window=10).max()

    fig, ax1 = plt.subplots()


    color = 'tab:red'
    ax1.set_xlabel('time (s)')
    ax1.set_ylabel('src2dst_pps', color=color)
    ax1.plot(t, data1, color=color)
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

    color = 'tab:blue'
    ax2.set_ylabel('dst2src_pps', color=color)  # we already handled the x-label with ax1
    ax2.plot(t, data2, color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    plt.xticks(range(0,round(max(t)), 10),rotation=45)
    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    # plt.show()
    fig.savefig(os.path.join('/Users/<user>/PhD/qoe/short_max_png', filename.replace('.csv', '.png')))   # save the figure to file
    plt.close(fig)    # close the figure window

In [ ]:
filepath = '/Users/<user>/PhD/github/qoe-research/beni-csv/window-0.csv'
for y1, y2 in [
    ('src2dst_avg_pkt_size_max', 'dst2src_avg_pkt_size_max'),
    ('src2dst_avg_pkt_size_min', 'dst2src_avg_pkt_size_min'),
    ('src2dst_avg_pkt_size_mean', 'dst2src_avg_pkt_size_mean'),
    ('src2dst_pps_max', 'dst2src_pps_max'),
    ('src2dst_pps_min', 'dst2src_pps_min'),
    ('src2dst_pps_mean', 'dst2src_pps_mean')
]:
    df = pd.read_csv(filepath)
    df = df[df['pkt_dir_ratio'] > 0.1]
    t = df['pktIdx'] * 2
    data1 = df[y1]
    data2 = df[y2]

    fig, ax1 = plt.subplots(figsize=(10, 2))  # set the figsize parameter to (10, 5) for a wider plot

    color = 'tab:red'
    ax1.set_xlabel('time (s)')
    ax1.set_ylabel(y1, color=color)
    ax1.plot(t, data1, color=color)
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

    color = 'tab:blue'
    ax2.set_ylabel(y2, color=color)  # we already handled the x-label with ax1
    ax2.plot(t, data2, color=color)
    ax2.tick_params(axis='y', labelcolor=color)

    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    plt.show()
    fig.savefig(os.path.join('/Users/<user>/PhD/github/qoe-research/png', y1.replace('src2dst_', '') + '.png'))   # save the figure to file
    plt.close(fig)    # close the figure window


In [ ]:
df = pd.read_csv('/Users/<user>/PhD/github/qoe-research/short/1.csv')
df = df[df['pkt_dir_ratio'] > 0.1]
t = df['pktIdx']
data1 = df['src2dst_avg_pkt_size']
data2 = df['dst2src_avg_pkt_size']

fig, ax1 = plt.subplots(figsize=(10, 2))  # set the figsize parameter to (10, 5) for a wider plot

color = 'tab:red'
ax1.set_xlabel('time (s)')
ax1.set_ylabel('src2dst_avg_pkt_size', color=color)
ax1.plot(t, data1, color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('dst2src_avg_pkt_size', color=color)  # we already handled the x-label with ax1
ax2.plot(t, data2, color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.show()